In [1]:
# ============================================================
# CATBOOST + MOWCA + LIME XAI
# SCI-PAPER READY VERSION
# FINAL CORRECTED VERSION
#
# Target: Cylinder compressive strength (MPa)
#
# MAIN FEATURES
# 1. Fixed 80/20 train-test split
# 2. Test set completely untouched during model development
# 3. Training-only median imputation
# 4. Fold-specific median imputation during repeated CV
# 5. CatBoost with MOWCA-tuned hyperparameters
# 6. Repeated 5-fold CV x 5 repetitions
# 7. 2,000-bootstrap confidence intervals for test metrics
# 8. Global LIME using ALL test observations
# 9. Zero contribution assigned to omitted LIME features
# 10. LIME feature-selection frequency
# 11. LIME stability across five random seeds
# 12. Mean, SD and CV of LIME contributions
# 13. Spearman rank stability across seeds
# 14. Minimum, median-nearest and maximum error explanations
# 15. Colorful publication-quality figures
# 16. 600 DPI output
# 17. Model, predictions, metrics and XAI tables saved
# 18. Package versions saved for reproducibility
# ============================================================


import os
import sys
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from lime.lime_tabular import LimeTabularExplainer

from sklearn.model_selection import (
    train_test_split,
    RepeatedKFold
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

import sklearn
import catboost
import lime


warnings.filterwarnings("ignore")


# ============================================================
# 1. SETTINGS
# ============================================================

DATA_PATH = (
    r"D:\2026 Work\My Papers\SCM-based concrete"
    r"\Modelling of the data\Data\Data.csv"
)

TARGET = "Cylinder compressive strength (MPa)"

TEST_SIZE = 0.20

RANDOM_STATE = 42


# ============================================================
# 2. CATBOOST SETTINGS (MOWCA-TUNED)
# ============================================================
#
# These hyperparameters were optimized using MOWCA
# (Multi-Objective Water Cycle Algorithm) on training data.
#
# MOWCA tuning results:
# learning_rate      = 0.20187322  [0.01, 0.3]
# depth              = 6           [3, 10]
# l2_leaf_reg        = 4.925521415 [1, 10]
# bagging_temperature= 0.219264541 [0, 1]
# random_strength    = 2.873836707 [0.5, 5]
# border_count       = 216         [32, 255]
# rsm                = 0.520345356 [0.5, 1]
# ============================================================

CATBOOST_PARAMS = {

    "learning_rate": 0.20187322,

    "depth": 6,

    "l2_leaf_reg": 4.925521415,

    "bagging_temperature": 0.219264541,

    "random_strength": 2.873836707,

    "border_count": 216,

    "rsm": 0.520345356,

    "iterations": 500,

    "random_seed": 42,

    "thread_count": 1,

    "verbose": False,

    "loss_function": "RMSE",

    "eval_metric": "RMSE"
}


# ============================================================
# 3. LIME SETTINGS
# ============================================================

LIME_NUM_SAMPLES = 5000

LIME_NUM_FEATURES = 15

GLOBAL_LIME_SAMPLES = None

LIME_STABILITY_SEEDS = [
    42,
    52,
    62,
    72,
    82
]

BASELINE_LIME_SEED = 42


# ============================================================
# 4. CROSS-VALIDATION SETTINGS
# ============================================================

CV_SPLITS = 5

CV_REPEATS = 5

CV_RANDOM_STATE = 42


# ============================================================
# 5. BOOTSTRAP SETTINGS
# ============================================================

BOOTSTRAP_ITERATIONS = 2000

BOOTSTRAP_RANDOM_STATE = 42


# ============================================================
# 6. REPRESENTATIVE OBSERVATIONS
# ============================================================

REPRESENTATIVE_METHODS = [

    "MIN_ERROR",

    "MEDIAN_ERROR",

    "MAX_ERROR"
]


# ============================================================
# 7. PUBLICATION SETTINGS
# ============================================================

DPI = 600

FIG_WIDTH = 11

FIG_HEIGHT = 8

FONT = "Times New Roman"

TOP_N_FEATURES_PLOT = 15


# ============================================================
# 8. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path(
    r"C:\Users\suhai\Desktop\LIME"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 9. COLOR PALETTE
# ============================================================

COLOR_PRIMARY = "#2563EB"

COLOR_SECONDARY = "#7C3AED"

COLOR_POSITIVE = "#16A34A"

COLOR_NEGATIVE = "#DC2626"

COLOR_ORANGE = "#EA580C"

COLOR_TEAL = "#0891B2"

COLOR_GOLD = "#CA8A04"

COLOR_GRAY = "#6B7280"


# ============================================================
# 10. MATPLOTLIB SETTINGS
# ============================================================

plt.rcParams["font.family"] = FONT

plt.rcParams["font.size"] = 16

plt.rcParams["axes.titlesize"] = 20

plt.rcParams["axes.labelsize"] = 18

plt.rcParams["xtick.labelsize"] = 15

plt.rcParams["ytick.labelsize"] = 15

plt.rcParams["axes.linewidth"] = 1.4

plt.rcParams["axes.titleweight"] = "bold"

plt.rcParams["axes.labelweight"] = "bold"

plt.rcParams["figure.dpi"] = DPI

plt.rcParams["savefig.dpi"] = DPI

plt.rcParams["savefig.bbox"] = "tight"


# ============================================================
# 11. LOAD DATA
# ============================================================

print("\nLoading dataset...")

df = pd.read_csv(DATA_PATH)

print(
    "Original dataset shape:",
    df.shape
)


if TARGET not in df.columns:

    raise ValueError(
        f"Target column '{TARGET}' was not found."
    )


# Remove rows with missing target

df = (
    df
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)


X = df.drop(
    columns=[TARGET]
).copy()


y = df[TARGET].copy()


# ============================================================
# 12. DATA TYPE CHECK
# ============================================================

for col in X.columns:

    if X[col].dtype == bool:

        X[col] = X[col].astype(int)


non_numeric = X.select_dtypes(
    exclude=[np.number]
).columns.tolist()


if len(non_numeric) > 0:

    raise ValueError(
        "Non-numeric columns detected: "
        + str(non_numeric)
    )


FEATURE_NAMES = X.columns.tolist()


print(
    "\nNumber of observations:",
    len(X)
)

print(
    "Number of input features:",
    len(FEATURE_NAMES)
)


print("\nFeatures:")

for feature in FEATURE_NAMES:

    print(
        "  ",
        feature
    )


# ============================================================
# 13. MISSING DATA REPORT
# ============================================================

missing_rate = (
    X
    .isna()
    .mean()
    .sort_values(
        ascending=False
    )
)


n_missing_features = int(
    (missing_rate > 0).sum()
)


total_missing_values = int(
    X.isna().sum().sum()
)


print(
    "\nFeatures with missing values:",
    f"{n_missing_features} of {len(FEATURE_NAMES)}"
)


print(
    "Total missing predictor values:",
    total_missing_values
)


if n_missing_features > 0:

    print(
        "\nTop missingness rates:"
    )

    print(
        missing_rate[
            missing_rate > 0
        ].head(10)
    )


# ============================================================
# 14. TRAIN-TEST SPLIT
# ============================================================

X_train_full, X_test_full, y_train, y_test = train_test_split(

    X,

    y,

    test_size=TEST_SIZE,

    random_state=RANDOM_STATE
)


print(
    "\nTraining samples:",
    len(X_train_full)
)

print(
    "Test samples:",
    len(X_test_full)
)


# ============================================================
# 15. FINAL MODEL IMPUTATION
# ============================================================

TRAIN_MEDIANS = (
    X_train_full
    .median()
)


X_train = (
    X_train_full
    .fillna(TRAIN_MEDIANS)
)


X_test = (
    X_test_full
    .fillna(TRAIN_MEDIANS)
)


X_train = pd.DataFrame(
    X_train,
    columns=FEATURE_NAMES
)


X_test = pd.DataFrame(
    X_test,
    columns=FEATURE_NAMES
)


# ============================================================
# 16. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    return {

        "R2":
            r2_score(
                y_true,
                y_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            ),

        "MAE":
            mean_absolute_error(
                y_true,
                y_pred
            ),

        "MAPE (%)":
            mean_absolute_percentage_error(
                y_true,
                y_pred
            ) * 100
    }


# ============================================================
# 17. TRAIN FINAL CATBOOST MODEL
# ============================================================

print(
    "\nTraining final CatBoost model with MOWCA-tuned hyperparameters..."
)


model = CatBoostRegressor(
    **CATBOOST_PARAMS
)


model.fit(
    X_train,
    y_train
)


print(
    "Final CatBoost model trained successfully."
)


# ============================================================
# 18. FINAL PREDICTIONS
# ============================================================

y_train_pred = (
    model.predict(
        X_train
    )
)


y_test_pred = (
    model.predict(
        X_test
    )
)


# ============================================================
# 19. FINAL METRICS
# ============================================================

train_metrics = calculate_metrics(
    y_train,
    y_train_pred
)


test_metrics = calculate_metrics(
    y_test,
    y_test_pred
)


print(
    "\n" + "=" * 70
)

print(
    "FINAL TEST-SET PERFORMANCE"
)

print(
    "=" * 70
)


for metric, value in test_metrics.items():

    print(
        f"{metric:<15}: {value:.4f}"
    )


print(
    "\nTraining performance:"
)


for metric, value in train_metrics.items():

    print(
        f"{metric:<15}: {value:.4f}"
    )


# ============================================================
# 20. BOOTSTRAP CONFIDENCE INTERVAL FUNCTION
# ============================================================

def bootstrap_metric_ci(
    y_true,
    y_pred,
    metric_name,
    n_bootstrap=2000,
    random_state=42
):

    rng = np.random.default_rng(
        random_state
    )

    y_true = np.asarray(
        y_true
    )

    y_pred = np.asarray(
        y_pred
    )

    n = len(y_true)

    bootstrap_values = []


    for _ in range(
        n_bootstrap
    ):

        indices = rng.integers(
            0,
            n,
            size=n
        )

        yt = y_true[
            indices
        ]

        yp = y_pred[
            indices
        ]


        if metric_name == "R2":

            value = r2_score(
                yt,
                yp
            )


        elif metric_name == "RMSE":

            value = np.sqrt(
                mean_squared_error(
                    yt,
                    yp
                )
            )


        elif metric_name == "MAE":

            value = mean_absolute_error(
                yt,
                yp
            )


        elif metric_name == "MAPE (%)":

            value = (
                mean_absolute_percentage_error(
                    yt,
                    yp
                ) * 100
            )


        else:

            raise ValueError(
                "Unknown metric."
            )


        bootstrap_values.append(
            value
        )


    bootstrap_values = np.asarray(
        bootstrap_values
    )


    lower = np.percentile(
        bootstrap_values,
        2.5
    )


    upper = np.percentile(
        bootstrap_values,
        97.5
    )


    return (
        lower,
        upper
    )


# ============================================================
# 21. BOOTSTRAP TEST CONFIDENCE INTERVALS
# ============================================================

print(
    "\nCalculating 2000-bootstrap confidence intervals..."
)


bootstrap_results = []


for metric in [
    "R2",
    "RMSE",
    "MAE",
    "MAPE (%)"
]:

    lower, upper = (
        bootstrap_metric_ci(

            y_test,

            y_test_pred,

            metric,

            BOOTSTRAP_ITERATIONS,

            BOOTSTRAP_RANDOM_STATE
        )
    )


    bootstrap_results.append({

        "Metric": metric,

        "Observed_Value":
            test_metrics[metric],

        "95%_CI_Lower":
            lower,

        "95%_CI_Upper":
            upper
    })


bootstrap_ci_df = pd.DataFrame(
    bootstrap_results
)


bootstrap_ci_df.to_csv(
    OUTPUT_DIR /
    "Test_Metrics_Bootstrap_95CI.csv",
    index=False
)


print(
    "\nBootstrap confidence intervals:"
)


for _, row in bootstrap_ci_df.iterrows():

    print(

        f"{row['Metric']:<12}: "
        f"{row['Observed_Value']:.4f} "
        f"("
        f"{row['95%_CI_Lower']:.4f}, "
        f"{row['95%_CI_Upper']:.4f}"
        f")"
    )


# ============================================================
# 22. OBSERVED VS PREDICTED PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


ax.scatter(

    y_test,

    y_test_pred,

    s=80,

    alpha=0.80,

    color=COLOR_PRIMARY,

    edgecolor="black",

    linewidth=0.6
)


min_value = min(
    y_test.min(),
    y_test_pred.min()
)


max_value = max(
    y_test.max(),
    y_test_pred.max()
)


ax.plot(

    [min_value, max_value],

    [min_value, max_value],

    linestyle="--",

    linewidth=2.5,

    color=COLOR_NEGATIVE,

    label="1:1 line"
)


ax.set_xlabel(
    "Observed Cylinder compressive strength (MPa)"
)


ax.set_ylabel(
    "Predicted Cylinder compressive strength (MPa)"
)


ax.set_title(
    "CatBoost: Observed vs Predicted"
)


ax.text(

    0.05,

    0.95,

    f"$R^2$ = {test_metrics['R2']:.3f}\n"
    f"RMSE = {test_metrics['RMSE']:.3f}\n"
    f"MAE = {test_metrics['MAE']:.3f}",

    transform=ax.transAxes,

    verticalalignment="top",

    fontsize=15,

    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor=COLOR_PRIMARY,
        alpha=0.90
    )
)


ax.legend(
    frameon=False,
    fontsize=13
)


ax.grid(
    alpha=0.20
)


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR /
    "Observed_vs_Predicted.png",

    dpi=DPI
)


plt.close()


# ============================================================
# 23. RESIDUAL PLOT
# ============================================================

residuals = (
    y_test.values -
    y_test_pred
)


fig, ax = plt.subplots(
    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


ax.scatter(

    y_test_pred,

    residuals,

    s=75,

    alpha=0.80,

    color=COLOR_SECONDARY,

    edgecolor="black",

    linewidth=0.6
)


ax.axhline(

    0,

    linestyle="--",

    linewidth=2.0,

    color=COLOR_NEGATIVE
)


ax.set_xlabel(
    "Predicted Cylinder compressive strength (MPa)"
)


ax.set_ylabel(
    "Residual (Observed − Predicted)"
)


ax.set_title(
    "CatBoost Residual Analysis"
)


ax.grid(
    alpha=0.20
)


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR /
    "Residual_Analysis.png",

    dpi=DPI
)


plt.close()


# ============================================================
# 24. CATBOOST FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({

    "Feature":
        FEATURE_NAMES,

    "Importance":
        model.feature_importances_
})


feature_importance = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
)


feature_importance.to_csv(

    OUTPUT_DIR /
    "CatBoost_Feature_Importance.csv",

    index=False
)


# ============================================================
# 25. CATBOOST FEATURE IMPORTANCE PLOT
# ============================================================

fi_plot = (
    feature_importance
    .head(TOP_N_FEATURES_PLOT)
    .sort_values(
        "Importance"
    )
)


fig, ax = plt.subplots(
    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


ax.barh(

    fi_plot["Feature"],

    fi_plot["Importance"],

    color=COLOR_TEAL,

    edgecolor="black",

    linewidth=0.5
)


ax.set_xlabel(
    "CatBoost Feature Importance"
)


ax.set_ylabel(
    "Feature"
)


ax.set_title(
    "CatBoost Feature Importance (MOWCA-Tuned)"
)


ax.grid(
    axis="x",
    alpha=0.20
)


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR /
    "CatBoost_Feature_Importance.png",

    dpi=DPI
)


plt.close()


# ============================================================
# 26. REPEATED 5-FOLD CV
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "REPEATED 5-FOLD CROSS-VALIDATION"
)

print(
    "Nested fold-specific median imputation"
)

print(
    "=" * 70
)


rkf = RepeatedKFold(

    n_splits=CV_SPLITS,

    n_repeats=CV_REPEATS,

    random_state=CV_RANDOM_STATE
)


cv_results = []


fold_number = 0


for train_idx, val_idx in rkf.split(
    X_train_full
):

    fold_number += 1


    X_cv_train_raw = (
        X_train_full.iloc[
            train_idx
        ]
    )


    X_cv_val_raw = (
        X_train_full.iloc[
            val_idx
        ]
    )


    y_cv_train = (
        y_train.iloc[
            train_idx
        ]
    )


    y_cv_val = (
        y_train.iloc[
            val_idx
        ]
    )


    # Fold-specific imputation

    fold_medians = (
        X_cv_train_raw
        .median()
    )


    X_cv_train = (
        X_cv_train_raw
        .fillna(
            fold_medians
        )
    )


    X_cv_val = (
        X_cv_val_raw
        .fillna(
            fold_medians
        )
    )


    cv_model = CatBoostRegressor(
        **CATBOOST_PARAMS
    )


    cv_model.fit(

        X_cv_train,

        y_cv_train
    )


    y_cv_pred = (
        cv_model.predict(
            X_cv_val
        )
    )


    fold_metrics = calculate_metrics(

        y_cv_val,

        y_cv_pred
    )


    fold_metrics["Fold"] = (
        fold_number
    )


    fold_metrics["Repeat"] = (
        int(
            (fold_number - 1)
            / CV_SPLITS
        ) + 1
    )


    fold_metrics["Fold_in_Repeat"] = (
        ((fold_number - 1)
         % CV_SPLITS) + 1
    )


    cv_results.append(
        fold_metrics
    )


    print(

        f"Fold {fold_number:02d}: "

        f"R2={fold_metrics['R2']:.4f}, "

        f"RMSE={fold_metrics['RMSE']:.4f}, "

        f"MAE={fold_metrics['MAE']:.4f}"
    )


cv_results_df = pd.DataFrame(
    cv_results
)


cv_results_df.to_csv(

    OUTPUT_DIR /
    "Repeated_5Fold_CV_Results.csv",

    index=False
)


# ============================================================
# 27. CV SUMMARY
# ============================================================

cv_summary = pd.DataFrame({

    "Metric": [

        "R2",

        "RMSE",

        "MAE",

        "MAPE (%)"
    ],

    "Mean": [

        cv_results_df["R2"].mean(),

        cv_results_df["RMSE"].mean(),

        cv_results_df["MAE"].mean(),

        cv_results_df["MAPE (%)"].mean()
    ],

    "SD": [

        cv_results_df["R2"].std(),

        cv_results_df["RMSE"].std(),

        cv_results_df["MAE"].std(),

        cv_results_df["MAPE (%)"].std()
    ],

    "Minimum": [

        cv_results_df["R2"].min(),

        cv_results_df["RMSE"].min(),

        cv_results_df["MAE"].min(),

        cv_results_df["MAPE (%)"].min()
    ],

    "Maximum": [

        cv_results_df["R2"].max(),

        cv_results_df["RMSE"].max(),

        cv_results_df["MAE"].max(),

        cv_results_df["MAPE (%)"].max()
    ]
})


cv_summary.to_csv(

    OUTPUT_DIR /
    "Repeated_CV_Summary.csv",

    index=False
)


print(
    "\nRepeated CV summary:"
)


for _, row in cv_summary.iterrows():

    print(

        f"{row['Metric']:<12}: "
        f"{row['Mean']:.4f} "
        f"± {row['SD']:.4f}"
    )


# ============================================================
# 28. CV PERFORMANCE PLOT
# ============================================================

fig, ax = plt.subplots(

    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


metric_order = [
    "R2",
    "RMSE",
    "MAE"
]


metric_colors = [
    COLOR_PRIMARY,
    COLOR_ORANGE,
    COLOR_POSITIVE
]


means = [

    cv_results_df[m].mean()

    for m in metric_order
]


sds = [

    cv_results_df[m].std()

    for m in metric_order
]


x = np.arange(
    len(metric_order)
)


ax.bar(

    x,

    means,

    yerr=sds,

    capsize=6,

    color=metric_colors,

    edgecolor="black",

    linewidth=0.7
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    metric_order
)


ax.set_ylabel(
    "CV Performance"
)


ax.set_title(
    "Repeated 5-Fold Cross-Validation Performance"
)


ax.grid(
    axis="y",
    alpha=0.20
)


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR /
    "Repeated_CV_Performance.png",

    dpi=DPI
)


plt.close()


# ============================================================
# 29. LIME EXPLAINER
# ============================================================

def create_lime_explainer(

    training_data,

    random_state

):

    return LimeTabularExplainer(

        training_data=
        training_data.values,

        feature_names=
        FEATURE_NAMES,

        mode="regression",

        discretize_continuous=True,

        random_state=random_state
    )


# ============================================================
# 30. LIME PREDICTION FUNCTION
# ============================================================

def lime_predict(data):

    data_df = pd.DataFrame(

        data,

        columns=FEATURE_NAMES
    )


    return model.predict(
        data_df
    )


# ============================================================
# 31. GLOBAL LIME OBSERVATIONS
# ============================================================

if GLOBAL_LIME_SAMPLES is None:

    lime_indices = np.arange(
        len(X_test)
    )

else:

    n_samples = min(

        GLOBAL_LIME_SAMPLES,

        len(X_test)
    )


    rng = np.random.default_rng(
        RANDOM_STATE
    )


    lime_indices = rng.choice(

        len(X_test),

        size=n_samples,

        replace=False
    )


print(

    "\nNumber of test observations "
    "used for global LIME:",

    len(lime_indices)
)


# ============================================================
# 32. BASELINE LIME
# ============================================================

print(
    "\nRunning baseline LIME..."
)


baseline_explainer = create_lime_explainer(

    X_train,

    BASELINE_LIME_SEED
)


baseline_records = []


global_contribution_matrix = np.zeros(

    (
        len(lime_indices),

        len(FEATURE_NAMES)
    )
)


global_selection_matrix = np.zeros(

    (
        len(lime_indices),

        len(FEATURE_NAMES)
    )
)


for counter, idx in enumerate(
    lime_indices
):

    row = X_test.iloc[idx]


    explanation = (
        baseline_explainer
        .explain_instance(

            row.values,

            lime_predict,

            num_features=
            LIME_NUM_FEATURES,

            num_samples=
            LIME_NUM_SAMPLES
        )
    )


    local_exp = next(

        iter(
            explanation.local_exp.values()
        )
    )


    for feature_idx, contribution in local_exp:

        global_contribution_matrix[
            counter,
            feature_idx
        ] = contribution


        global_selection_matrix[
            counter,
            feature_idx
        ] = 1


        baseline_records.append({

            "Test_Index":
                int(idx),

            "Feature":
                FEATURE_NAMES[
                    feature_idx
                ],

            "Contribution":
                contribution,

            "Absolute_Contribution":
                abs(contribution),

            "Selected":
                1,

            "Seed":
                BASELINE_LIME_SEED
        })


    if (
        (counter + 1) % 10 == 0
    ):

        print(

            f"Processed "
            f"{counter + 1}/"
            f"{len(lime_indices)}"
        )


baseline_lime_df = pd.DataFrame(
    baseline_records
)


baseline_lime_df.to_csv(

    OUTPUT_DIR /
    "LIME_Local_Contributions.csv",

    index=False
)


# ============================================================
# 33. COMPLETE GLOBAL LIME DATASET
# ============================================================

complete_global_records = []


for row_number, idx in enumerate(
    lime_indices
):

    for feature_idx, feature_name in enumerate(
        FEATURE_NAMES
    ):

        contribution = (
            global_contribution_matrix[
                row_number,
                feature_idx
            ]
        )


        selected = (
            global_selection_matrix[
                row_number,
                feature_idx
            ]
        )


        complete_global_records.append({

            "Test_Index":
                int(idx),

            "Feature":
                feature_name,

            "Contribution":
                contribution,

            "Absolute_Contribution":
                abs(contribution),

            "Selected":
                int(selected)
        })


complete_global_df = pd.DataFrame(
    complete_global_records
)


complete_global_df.to_csv(

    OUTPUT_DIR /
    "LIME_Global_Complete_Matrix.csv",

    index=False
)


# ============================================================
# 34. GLOBAL LIME IMPORTANCE
# ============================================================

global_lime = (

    complete_global_df

    .groupby("Feature")

    .agg(

        Mean_Absolute_Contribution=(

            "Absolute_Contribution",

            "mean"
        ),

        SD_Absolute_Contribution=(

            "Absolute_Contribution",

            "std"
        ),

        Mean_Contribution=(

            "Contribution",

            "mean"
        ),

        SD_Contribution=(

            "Contribution",

            "std"
        ),

        Median_Absolute_Contribution=(

            "Absolute_Contribution",

            "median"
        ),

        Explanation_Count=(

            "Selected",

            "sum"
        ),

        Total_Observations=(

            "Selected",

            "count"
        )
    )

    .reset_index()
)


global_lime[
    "Selection_Frequency_Percent"
] = (

    global_lime[
        "Explanation_Count"
    ]

    /

    global_lime[
        "Total_Observations"
    ]

) * 100


global_lime = (

    global_lime

    .sort_values(

        "Mean_Absolute_Contribution",

        ascending=False
    )
)


global_lime.to_csv(

    OUTPUT_DIR /
    "Global_LIME_Feature_Importance.csv",

    index=False
)


# ============================================================
# 35. GLOBAL LIME PLOT
# ============================================================

plot_data = (

    global_lime

    .head(
        TOP_N_FEATURES_PLOT
    )

    .sort_values(
        "Mean_Absolute_Contribution"
    )
)


fig, ax = plt.subplots(

    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


ax.barh(

    plot_data["Feature"],

    plot_data[
        "Mean_Absolute_Contribution"
    ],

    color=COLOR_PRIMARY,

    edgecolor="black",

    linewidth=0.6
)


ax.set_xlabel(
    "Mean Absolute LIME Contribution"
)


ax.set_ylabel(
    "Feature"
)


ax.set_title(
    "Global LIME Feature Importance"
)


ax.grid(
    axis="x",
    alpha=0.20
)


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR /
    "Global_LIME_Feature_Importance.png",

    dpi=DPI
)


plt.close()


# ============================================================
# 36. LIME SELECTION FREQUENCY PLOT
# ============================================================

selection_plot = (

    global_lime

    .sort_values(
        "Selection_Frequency_Percent"
    )

    .tail(
        TOP_N_FEATURES_PLOT
    )
)


fig, ax = plt.subplots(

    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


ax.barh(

    selection_plot["Feature"],

    selection_plot[
        "Selection_Frequency_Percent"
    ],

    color=COLOR_GOLD,

    edgecolor="black",

    linewidth=0.6
)


ax.set_xlabel(
    "LIME Selection Frequency (%)"
)


ax.set_ylabel(
    "Feature"
)


ax.set_title(
    "LIME Feature Selection Frequency"
)


ax.set_xlim(
    0,
    100
)


ax.grid(
    axis="x",
    alpha=0.20
)


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR /
    "LIME_Feature_Selection_Frequency.png",

    dpi=DPI
)


plt.close()


# ============================================================
# 37. LIME STABILITY ANALYSIS
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "LIME STABILITY ANALYSIS"
)

print(
    "Re-running LIME explainer with 5 different seeds"
)

print(
    "=" * 70
)


stability_records = []


seed_feature_matrices = {}


for seed in LIME_STABILITY_SEEDS:

    print(
        f"\nRunning LIME stability seed = {seed}"
    )


    if seed == BASELINE_LIME_SEED:

        seed_matrix = (
            global_contribution_matrix.copy()
        )

        print(
            "Baseline seed reused."
        )


    else:

        seed_explainer = create_lime_explainer(

            X_train,

            seed
        )


        seed_matrix = np.zeros(

            (
                len(lime_indices),

                len(FEATURE_NAMES)
            )
        )


        for counter, idx in enumerate(
            lime_indices
        ):

            row = X_test.iloc[idx]


            explanation = (
                seed_explainer
                .explain_instance(

                    row.values,

                    lime_predict,

                    num_features=
                    LIME_NUM_FEATURES,

                    num_samples=
                    LIME_NUM_SAMPLES
                )
            )


            local_exp = next(

                iter(
                    explanation.local_exp.values()
                )
            )


            for feature_idx, contribution in local_exp:

                seed_matrix[
                    counter,
                    feature_idx
                ] = contribution


            if (
                (counter + 1) % 10 == 0
            ):

                print(

                    f"  Seed {seed}: "
                    f"{counter + 1}/"
                    f"{len(lime_indices)}"
                )


    seed_feature_matrices[
        seed
    ] = seed_matrix


    # Save complete seed-level records

    for row_number, idx in enumerate(
        lime_indices
    ):

        for feature_idx, feature_name in enumerate(
            FEATURE_NAMES
        ):

            contribution = (
                seed_matrix[
                    row_number,
                    feature_idx
                ]
            )


            stability_records.append({

                "Seed":
                    seed,

                "Test_Index":
                    int(idx),

                "Feature":
                    feature_name,

                "Contribution":
                    contribution,

                "Absolute_Contribution":
                    abs(contribution)
            })


lime_stability_df = pd.DataFrame(
    stability_records
)


lime_stability_df.to_csv(

    OUTPUT_DIR /
    "LIME_Stability_Raw.csv",

    index=False
)


# ============================================================
# 38. LIME STABILITY SUMMARY
# ============================================================

lime_stability_summary = (

    lime_stability_df

    .groupby("Feature")

    .agg(

        Mean_Absolute_Contribution=(

            "Absolute_Contribution",

            "mean"
        ),

        SD_Absolute_Contribution=(

            "Absolute_Contribution",

            "std"
        ),

        Mean_Contribution=(

            "Contribution",

            "mean"
        ),

        SD_Contribution=(

            "Contribution",

            "std"
        )
    )

    .reset_index()
)


lime_stability_summary[
    "CV_Percent"
] = (

    lime_stability_summary[
        "SD_Absolute_Contribution"
    ]

    /

    lime_stability_summary[
        "Mean_Absolute_Contribution"
    ].replace(
        0,
        np.nan
    )

) * 100


lime_stability_summary = (

    lime_stability_summary

    .sort_values(

        "Mean_Absolute_Contribution",

        ascending=False
    )
)


lime_stability_summary.to_csv(

    OUTPUT_DIR /
    "LIME_Stability_Summary.csv",

    index=False
)


# ============================================================
# 39. LIME RANK STABILITY
# ============================================================

per_seed_feature_importance = {}


for seed, matrix in seed_feature_matrices.items():

    per_seed_feature_importance[seed] = (

        np.abs(matrix)

        .mean(
            axis=0
        )
    )


per_seed_df = pd.DataFrame(

    per_seed_feature_importance,

    index=FEATURE_NAMES
)


per_seed_df.index.name = (
    "Feature"
)


per_seed_df.to_csv(

    OUTPUT_DIR /
    "LIME_Per_Seed_Feature_Importance.csv"
)


rank_correlation_matrix = (
    per_seed_df
    .corr(
        method="spearman"
    )
)


rank_correlation_matrix.to_csv(

    OUTPUT_DIR /
    "LIME_Stability_Rank_Correlation.csv"
)


if rank_correlation_matrix.shape[0] > 1:

    mask = ~np.eye(

        rank_correlation_matrix.shape[0],

        dtype=bool
    )


    mean_rank_correlation = (

        rank_correlation_matrix.values[
            mask
        ].mean()
    )

else:

    mean_rank_correlation = np.nan


print(

    "\nMean pairwise Spearman rank "
    "correlation across LIME seeds:",

    f"{mean_rank_correlation:.4f}"
)


# ============================================================
# 40. LIME STABILITY PLOT
# ============================================================

stability_plot = (

    lime_stability_summary

    .head(
        TOP_N_FEATURES_PLOT
    )

    .sort_values(
        "Mean_Absolute_Contribution"
    )
)


fig, ax = plt.subplots(

    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


ax.barh(

    stability_plot["Feature"],

    stability_plot[
        "Mean_Absolute_Contribution"
    ],

    xerr=stability_plot[
        "SD_Absolute_Contribution"
    ],

    capsize=4,

    color=COLOR_SECONDARY,

    edgecolor="black",

    linewidth=0.5
)


ax.set_xlabel(

    "Mean Absolute LIME Contribution ± SD"
)


ax.set_ylabel(
    "Feature"
)


ax.set_title(

    "LIME Stability Across Random Seeds"
)


ax.grid(
    axis="x",
    alpha=0.20
)


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR /
    "LIME_Stability.png",

    dpi=DPI
)


plt.close()


# ============================================================
# 41. LIME RANK CORRELATION HEATMAP
# ============================================================

fig, ax = plt.subplots(

    figsize=(
        9,
        8
    )
)


im = ax.imshow(

    rank_correlation_matrix.values,

    cmap="viridis",

    vmin=0,

    vmax=1
)


ax.set_xticks(
    np.arange(
        len(rank_correlation_matrix.columns)
    )
)


ax.set_yticks(
    np.arange(
        len(rank_correlation_matrix.index)
    )
)


ax.set_xticklabels(
    rank_correlation_matrix.columns
)


ax.set_yticklabels(
    rank_correlation_matrix.index
)


for i in range(
    rank_correlation_matrix.shape[0]
):

    for j in range(
        rank_correlation_matrix.shape[1]
    ):

        ax.text(

            j,

            i,

            f"{rank_correlation_matrix.iloc[i, j]:.2f}",

            ha="center",

            va="center",

            fontsize=12,

            color="white"
        )


ax.set_xlabel(
    "LIME Seed"
)


ax.set_ylabel(
    "LIME Seed"
)


ax.set_title(
    "LIME Feature-Ranking Stability"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Spearman correlation"
)


plt.tight_layout()


plt.savefig(

    OUTPUT_DIR /
    "LIME_Rank_Stability_Heatmap.png",

    dpi=DPI
)


plt.close()


# ============================================================
# 42. LIME SEED SUMMARY
# ============================================================

seed_summary = []


for seed, matrix in seed_feature_matrices.items():

    seed_summary.append({

        "Seed":
            seed,

        "Mean_Absolute_Contribution":
            np.abs(matrix).mean(),

        "SD_Absolute_Contribution":
            np.abs(matrix).std()
    })


seed_summary_df = pd.DataFrame(
    seed_summary
)


seed_summary_df.to_csv(

    OUTPUT_DIR /
    "LIME_Seed_Summary.csv",

    index=False
)


# ============================================================
# 43. TEST PREDICTIONS TABLE
# ============================================================

test_predictions_df = pd.DataFrame({

    "Test_Index":
        np.arange(
            len(X_test)
        ),

    "Observed":
        y_test.values,

    "Predicted":
        y_test_pred,

    "Residual":
        y_test.values -
        y_test_pred,

    "Absolute_Error":
        np.abs(
            y_test.values -
            y_test_pred
        )
})


test_predictions_df[
    "Relative_Error_Percent"
] = np.where(

    np.abs(
        test_predictions_df[
            "Observed"
        ]
    ) > 0,

    (

        test_predictions_df[
            "Absolute_Error"
        ]

        /

        np.abs(
            test_predictions_df[
                "Observed"
            ]
        )

    ) * 100,

    np.nan
)


test_predictions_df.to_csv(

    OUTPUT_DIR /
    "Test_Predictions.csv",

    index=False
)


# ============================================================
# 44. REPRESENTATIVE OBSERVATIONS
# ============================================================

min_error_idx = (

    test_predictions_df[
        "Absolute_Error"
    ]

    .idxmin()
)


median_error_value = (

    test_predictions_df[
        "Absolute_Error"
    ]

    .median()
)


median_error_idx = (

    test_predictions_df[
        "Absolute_Error"
    ]

    -

    median_error_value

).abs().idxmin()


max_error_idx = (

    test_predictions_df[
        "Absolute_Error"
    ]

    .idxmax()
)


representative_indices = {

    "MIN_ERROR":

        int(
            test_predictions_df.loc[
                min_error_idx,
                "Test_Index"
            ]
        ),

    "MEDIAN_ERROR":

        int(
            test_predictions_df.loc[
                median_error_idx,
                "Test_Index"
            ]
        ),

    "MAX_ERROR":

        int(
            test_predictions_df.loc[
                max_error_idx,
                "Test_Index"
            ]
        )
}


print(
    "\nRepresentative observations:"
)


for method, idx in representative_indices.items():

    row = (

        test_predictions_df[
            test_predictions_df[
                "Test_Index"
            ] == idx
        ]

        .iloc[0]
    )


    print(

        f"{method:<15}: "

        f"Index={idx}, "

        f"Observed={row['Observed']:.4f}, "

        f"Predicted={row['Predicted']:.4f}, "

        f"Error={row['Absolute_Error']:.4f}"
    )


# ============================================================
# 45. LOCAL LIME EXPLANATIONS
# ============================================================

local_summary_records = []


for method, idx in representative_indices.items():

    print(

        f"\nGenerating local LIME explanation: "
        f"{method}"
    )


    row = X_test.iloc[idx]


    explanation = (

        baseline_explainer
        .explain_instance(

            row.values,

            lime_predict,

            num_features=
            LIME_NUM_FEATURES,

            num_samples=
            LIME_NUM_SAMPLES
        )
    )


    # Human-readable LIME rules

    explanation_list = (
        explanation.as_list()
    )


    readable_df = pd.DataFrame(

        explanation_list,

        columns=[
            "Feature_Rule",
            "Contribution"
        ]
    )


    readable_df[
        "Absolute_Contribution"
    ] = (

        readable_df[
            "Contribution"
        ].abs()
    )


    readable_df.to_csv(

        OUTPUT_DIR /
        f"LIME_{method}_Readable_Rules.csv",

        index=False
    )


    # Original feature names

    local_exp = next(

        iter(
            explanation.local_exp.values()
        )
    )


    original_feature_records = []


    for feature_idx, contribution in local_exp:

        feature_name = (
            FEATURE_NAMES[
                feature_idx
            ]
        )


        original_feature_records.append({

            "Feature":
                feature_name,

            "Contribution":
                contribution,

            "Absolute_Contribution":
                abs(contribution)
        })


    original_feature_df = pd.DataFrame(

        original_feature_records
    )


    original_feature_df = (

        original_feature_df

        .sort_values(

            "Absolute_Contribution",

            ascending=False
        )
    )


    original_feature_df.to_csv(

        OUTPUT_DIR /
        f"LIME_{method}_Original_Features.csv",

        index=False
    )


    observation_info = (

        test_predictions_df[

            test_predictions_df[
                "Test_Index"
            ] == idx

        ]

        .iloc[0]
    )


    for _, feature_row in original_feature_df.iterrows():

        local_summary_records.append({

            "Case":
                method,

            "Test_Index":
                idx,

            "Observed":
                observation_info[
                    "Observed"
                ],

            "Predicted":
                observation_info[
                    "Predicted"
                ],

            "Absolute_Error":
                observation_info[
                    "Absolute_Error"
                ],

            "Feature":
                feature_row[
                    "Feature"
                ],

            "Contribution":
                feature_row[
                    "Contribution"
                ],

            "Absolute_Contribution":
                feature_row[
                    "Absolute_Contribution"
                ]
        })


# ============================================================
# 46. SAVE LOCAL LIME SUMMARY
# ============================================================

local_summary_df = pd.DataFrame(
    local_summary_records
)


local_summary_df.to_csv(

    OUTPUT_DIR /
    "LIME_Local_Summary.csv",

    index=False
)


# ============================================================
# 47. LOCAL LIME PLOTS
# ============================================================

for method, idx in representative_indices.items():

    case_data = (

        local_summary_df[

            local_summary_df[
                "Case"
            ] == method

        ]

        .copy()
    )


    case_data = (

        case_data

        .sort_values(

            "Absolute_Contribution",

            ascending=False
        )

        .head(
            LIME_NUM_FEATURES
        )
    )


    case_data = (

        case_data

        .sort_values(
            "Contribution"
        )
    )


    colors = [

        COLOR_NEGATIVE
        if value < 0
        else COLOR_POSITIVE

        for value in
        case_data[
            "Contribution"
        ]
    ]


    fig, ax = plt.subplots(

        figsize=(
            FIG_WIDTH,
            FIG_HEIGHT
        )
    )


    ax.barh(

        case_data["Feature"],

        case_data["Contribution"],

        color=colors,

        edgecolor="black",

        linewidth=0.5
    )


    ax.axvline(

        0,

        linewidth=1.5,

        color="black"
    )


    ax.set_xlabel(
        "LIME Contribution"
    )


    ax.set_ylabel(
        "Feature"
    )


    ax.set_title(

        f"Local LIME Explanation: "
        f"{method}"
    )


    ax.grid(
        axis="x",
        alpha=0.20
    )


    plt.tight_layout()


    plt.savefig(

        OUTPUT_DIR /
        f"LIME_Local_{method}.png",

        dpi=DPI
    )


    plt.close()


# ============================================================
# 48. LIME HTML FILES
# ============================================================

for method, idx in representative_indices.items():

    row = X_test.iloc[idx]


    html_explainer = create_lime_explainer(

        X_train,

        BASELINE_LIME_SEED
    )


    explanation = (

        html_explainer

        .explain_instance(

            row.values,

            lime_predict,

            num_features=
            LIME_NUM_FEATURES,

            num_samples=
            LIME_NUM_SAMPLES
        )
    )


    explanation.save_to_file(

        str(

            OUTPUT_DIR /

            f"LIME_{method}.html"
        )
    )


# ============================================================
# 49. SAVE TRAINING MEDIANS
# ============================================================

TRAIN_MEDIANS.to_csv(

    OUTPUT_DIR /
    "Training_Medians.csv"
)


# ============================================================
# 50. SAVE MODEL
# ============================================================

model.save_model(

    str(

        OUTPUT_DIR /

        "CatBoost_CylinderStrength_MOWCA_Model.cbm"
    )
)


# ============================================================
# 51. SAVE CATBOOST PARAMETERS (MOWCA-TUNED)
# ============================================================

params_df = pd.DataFrame({

    "Parameter":
        list(
            CATBOOST_PARAMS.keys()
        ),

    "Value":
        [
            str(v)
            for v in
            CATBOOST_PARAMS.values()
        ],

    "Source":
        [
            "MOWCA-tuned" if k in [
                "learning_rate",
                "depth",
                "l2_leaf_reg",
                "bagging_temperature",
                "random_strength",
                "border_count",
                "rsm"
            ] else "Fixed"
            for k in CATBOOST_PARAMS.keys()
        ]
})


params_df.to_csv(

    OUTPUT_DIR /
    "CatBoost_MOWCA_Parameters.csv",

    index=False
)


# ============================================================
# 52. MOWCA OPTIMIZATION DETAILS
# ============================================================

mowca_details = pd.DataFrame({

    "Hyperparameter": [
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "bagging_temperature",
        "random_strength",
        "border_count",
        "rsm"
    ],

    "Optimized_Value": [
        0.20187322,
        6,
        4.925521415,
        0.219264541,
        2.873836707,
        216,
        0.520345356
    ],

    "Lower_Bound": [
        0.01,
        3,
        1,
        0,
        0.5,
        32,
        0.5
    ],

    "Upper_Bound": [
        0.3,
        10,
        10,
        1,
        5,
        255,
        1
    ],

    "Distance_to_Nearest_Bound": [
        0.09812678,
        3,
        3.925521415,
        0.219264541,
        2.126163293,
        39,
        0.020345356
    ]
})


mowca_details.to_csv(

    OUTPUT_DIR /
    "MOWCA_Optimization_Details.csv",

    index=False
)


# ============================================================
# 53. SOFTWARE VERSION INFORMATION
# ============================================================

version_df = pd.DataFrame({

    "Software": [

        "Python",

        "Platform",

        "NumPy",

        "Pandas",

        "Scikit-learn",

        "CatBoost",

        "LIME",

        "Matplotlib"
    ],

    "Version": [

        sys.version,

        platform.platform(),

        np.__version__,

        pd.__version__,

        sklearn.__version__,

        catboost.__version__,

        lime.__version__
        if hasattr(lime, "__version__")
        else "Not available",

        plt.matplotlib.__version__
    ]
})


version_df.to_csv(

    OUTPUT_DIR /
    "Software_Versions.csv",

    index=False
)


# ============================================================
# 54. FINAL METHODOLOGY SUMMARY
# ============================================================

summary = {

    "Dataset_Size":
        len(df),

    "Number_of_Features":
        len(FEATURE_NAMES),

    "Features_With_Missing_Values":
        n_missing_features,

    "Total_Missing_Predictor_Values":
        total_missing_values,

    "Training_Size":
        len(X_train),

    "Test_Size":
        len(X_test),

    "Target_Variable":
        TARGET,

    "Optimization_Method":
        "MOWCA (Multi-Objective Water Cycle Algorithm)",

    "Test_R2":
        test_metrics["R2"],

    "Test_RMSE":
        test_metrics["RMSE"],

    "Test_MAE":
        test_metrics["MAE"],

    "Test_MAPE (%)":
        test_metrics["MAPE (%)"],

    "CV_R2_Mean":
        cv_results_df["R2"].mean(),

    "CV_R2_SD":
        cv_results_df["R2"].std(),

    "CV_RMSE_Mean":
        cv_results_df["RMSE"].mean(),

    "CV_RMSE_SD":
        cv_results_df["RMSE"].std(),

    "CV_MAE_Mean":
        cv_results_df["MAE"].mean(),

    "CV_MAE_SD":
        cv_results_df["MAE"].std(),

    "CV_Folds":
        CV_SPLITS,

    "CV_Repetitions":
        CV_REPEATS,

    "CV_Total_Fits":
        len(cv_results_df),

    "CV_Imputation":
        "Fold-specific median",

    "Final_Model_Imputation":
        "Median from 80% training set",

    "Test_Set_Used_During_CV":
        "No",

    "LIME_Number_of_Seeds":
        len(
            LIME_STABILITY_SEEDS
        ),

    "LIME_Seeds_Used":
        str(LIME_STABILITY_SEEDS),

    "LIME_Samples_Per_Explanation":
        LIME_NUM_SAMPLES,

    "LIME_Selected_Features_Per_Explanation":
        LIME_NUM_FEATURES,

    "LIME_Global_Observations":
        len(lime_indices),

    "LIME_Mean_Rank_Correlation":
        mean_rank_correlation,

    "Bootstrap_Iterations":
        BOOTSTRAP_ITERATIONS,

    "Bootstrap_CI":
        "95%"
}


summary_df = pd.DataFrame(
    [summary]
)


summary_df.to_csv(

    OUTPUT_DIR /
    "Final_Methodology_Summary.csv",

    index=False
)


# ============================================================
# 55. FINAL REPORT
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "ANALYSIS COMPLETED"
)

print(
    "=" * 70
)


print(
    "\nAll outputs saved to:"
)


print(
    OUTPUT_DIR
)


print(
    "\nMain output files:"
)


main_files = [

    "Observed_vs_Predicted.png",

    "Residual_Analysis.png",

    "CatBoost_Feature_Importance.png",

    "Global_LIME_Feature_Importance.png",

    "LIME_Feature_Selection_Frequency.png",

    "LIME_Stability.png",

    "LIME_Rank_Stability_Heatmap.png",

    "Repeated_CV_Performance.png",

    "Test_Metrics_Bootstrap_95CI.csv",

    "Repeated_CV_Summary.csv",

    "Repeated_5Fold_CV_Results.csv",

    "Global_LIME_Feature_Importance.csv",

    "LIME_Stability_Summary.csv",

    "LIME_Stability_Rank_Correlation.csv",

    "LIME_Per_Seed_Feature_Importance.csv",

    "LIME_Local_Summary.csv",

    "Test_Predictions.csv",

    "CatBoost_Feature_Importance.csv",

    "CatBoost_MOWCA_Parameters.csv",

    "MOWCA_Optimization_Details.csv",

    "Training_Medians.csv",

    "Software_Versions.csv",

    "Final_Methodology_Summary.csv",

    "CatBoost_CylinderStrength_MOWCA_Model.cbm"
]


for number, filename in enumerate(
    main_files,
    start=1
):

    print(
        f"{number:02d}. {filename}"
    )


print(
    "\nFinal test R2:",
    f"{test_metrics['R2']:.4f}"
)


print(

    "Repeated CV R2:",

    f"{cv_results_df['R2'].mean():.4f}",

    "±",

    f"{cv_results_df['R2'].std():.4f}"
)


print(

    "Mean LIME rank correlation:",

    f"{mean_rank_correlation:.4f}"
)


print(
    "\nDone."
)


Loading dataset...
Original dataset shape: (1456, 9)

Number of observations: 1456
Number of input features: 8

Features:
    Cement(kg/m3)
    Water(kg/m3)
   Coarse aggregate(kg/m3)
   Fine aggregate(kg/m3)
    FA (kg/m3)
   SF (kg/m3)
   GGBFS (kg/m3)
   SP (kg/m3)

Features with missing values: 0 of 8
Total missing predictor values: 0

Training samples: 1164
Test samples: 292

Training final CatBoost model with MOWCA-tuned hyperparameters...
Final CatBoost model trained successfully.

FINAL TEST-SET PERFORMANCE
R2             : 0.8362
RMSE           : 7.1665
MAE            : 4.8083
MAPE (%)       : 11.2233

Training performance:
R2             : 0.9887
RMSE           : 1.9067
MAE            : 1.3448
MAPE (%)       : 3.2519

Calculating 2000-bootstrap confidence intervals...

Bootstrap confidence intervals:
R2          : 0.8362 (0.7833, 0.8782)
RMSE        : 7.1665 (6.2653, 7.9911)
MAE         : 4.8083 (4.2066, 5.4104)
MAPE (%)    : 11.2233 (9.8079, 12.7017)


findfont: Font family ['cmsy10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmr10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmtt10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmmi10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmb10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmss10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmex10'] not found. Falling back to DejaVu Sans.



REPEATED 5-FOLD CROSS-VALIDATION
Nested fold-specific median imputation
Fold 01: R2=0.8676, RMSE=6.1976, MAE=4.2874
Fold 02: R2=0.8693, RMSE=6.8137, MAE=4.6564
Fold 03: R2=0.8509, RMSE=6.9513, MAE=4.8110
Fold 04: R2=0.8899, RMSE=6.0205, MAE=3.9819
Fold 05: R2=0.8351, RMSE=7.0737, MAE=4.8841
Fold 06: R2=0.8316, RMSE=6.8050, MAE=4.8744
Fold 07: R2=0.8571, RMSE=6.9865, MAE=4.4367
Fold 08: R2=0.8845, RMSE=6.2038, MAE=4.4505
Fold 09: R2=0.8859, RMSE=5.7916, MAE=4.0055
Fold 10: R2=0.8494, RMSE=7.1831, MAE=4.6352
Fold 11: R2=0.8770, RMSE=6.1864, MAE=4.4039
Fold 12: R2=0.8572, RMSE=6.9737, MAE=4.8987
Fold 13: R2=0.8361, RMSE=6.8266, MAE=4.6064
Fold 14: R2=0.8458, RMSE=7.0533, MAE=4.7832
Fold 15: R2=0.9022, RMSE=5.7832, MAE=4.1729
Fold 16: R2=0.8895, RMSE=5.8740, MAE=4.2872
Fold 17: R2=0.9151, RMSE=5.7032, MAE=4.0990
Fold 18: R2=0.8703, RMSE=6.7589, MAE=4.5744
Fold 19: R2=0.8730, RMSE=6.0083, MAE=4.1376
Fold 20: R2=0.8417, RMSE=6.4874, MAE=4.3031
Fold 21: R2=0.8554, RMSE=6.5290, MAE=4.6458
Fol